<a href="https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q datasets huggingface_hub duckdb pandas pyarrow

In [5]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi()
print(api.whoami(token=HF_TOKEN))

{'type': 'user', 'id': '6a609943ec6acc0423acb037', 'name': 'adityaa311', 'fullname': 'Aditya Rajesh Dabhade', 'isPro': False, 'avatarUrl': '/avatars/33bb4ed6450474c6dfd8c84afc6a4078.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HF_TOKEN', 'role': 'fineGrained', 'createdAt': '2026-07-31T08:44:07.435Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6a609943ec6acc0423acb037', 'type': 'user', 'name': 'adityaa311'}, 'permissions': ['repo.content.read']}]}}}}


In [6]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi()

info = api.dataset_info(
    repo_id="FlyRank/internship-warehouse",
    token=HF_TOKEN,
)

print(info.id)

FlyRank/internship-warehouse


In [7]:
from huggingface_hub import snapshot_download
import os

repo_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
)

print(repo_path)

print("\nFiles:")
for root, dirs, files in os.walk(repo_path):
    for file in files:
        print(os.path.join(root, file))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2

Files:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/.gitattributes
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/README.md
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_query_90d.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet
/root/.cac

In [8]:
import duckdb
import os

con = duckdb.connect()

base = repo_path

con.execute(f"""
CREATE VIEW fact_daily AS
SELECT *
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet');
""")

con.execute(f"""
CREATE VIEW dim_content AS
SELECT *
FROM read_parquet('{base}/dim_content.parquet');
""")

con.execute(f"""
CREATE VIEW dim_clients AS
SELECT *
FROM read_parquet('{base}/dim_clients.parquet');
""")

print("✅ Views created successfully")

✅ Views created successfully


In [9]:
con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_daily


In [10]:
con.sql("DESCRIBE fact_daily").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
con.sql("SELECT * FROM fact_daily LIMIT 5").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [12]:
con.sql("DESCRIBE dim_content").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [13]:
con.sql("SELECT * FROM dim_content LIMIT 5").df()

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


## 1. Unit of analysis + time window

- **Unit of analysis:** One row represents the daily search performance of one content page (`content_hash_id`) for one client (`client_hash_id`) on a single reporting date.
- **Tables used:** `fact_daily` as the primary fact table and `dim_content` for additional content metadata if required.
- **Time window:** March 2026 (`2026-03`). This is a mid-panel month and avoids using the final month for development.
- **Prediction / Proxy:** Predict whether a content page should be prioritized for refresh based on historical search performance.
- **Excluded:** Data from the final month (June 2026) and any future information that would not be available at the time the refresh decision is made.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM fact_daily
""").df()

,total_rows,unique_content,start_date,end_date
0,9841378,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- gsc_data_available

### Label / Proxy
Refresh priority (proxy): Pages with declining search performance that should be considered for content refresh.

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
- Data from the final month (June 2026), because it should remain a sealed evaluation period.
- Any information that would only be known after the refresh decision is made.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_frame = con.sql("""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    gsc_data_available
FROM fact_daily
LIMIT 10
""").df()

feature_frame

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,gsc_data_available
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,True
1,2026-03-01,content_05597932fe4da067,1,0,0,True,True
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,True
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,True
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,True
5,2026-03-01,content_36c36abc7650d7af,239,1,1756,True,True
6,2026-03-01,content_a7da352b73b02668,191,0,1496,True,True
7,2026-03-01,content_05434271b257bb68,55,0,180,True,True
8,2026-03-01,content_d056587ff7faca0c,77,0,434,True,True
9,2026-03-01,content_bfd1e41c2af250c8,2,0,9,True,True


## 3. Verify it with queries (grain, counts, missing values, windows)

## Verification Queries

The following queries verify the contract.

1. Verify the grain of the dataset.
2. Verify the row count and date range.
3. Verify available Search Console data using `IS TRUE`.

In [17]:
print("="*60)
print("Query 1: Verify the Grain")
print("="*60)

grain = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS records
FROM fact_daily
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

print(grain)

print("\n")
print("="*60)
print("Query 2: Row Count and Date Span")
print("="*60)

row_info = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM fact_daily;
""").df()

print(row_info)

print("\n")
print("="*60)
print("Query 3: Availability Check (IS TRUE)")
print("="*60)

availability = con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM fact_daily
WHERE gsc_data_available IS TRUE;
""").df()

print(availability)

Query 1: Verify the Grain


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, records]
Index: []


Query 2: Row Count and Date Span
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31


Query 3: Availability Check (IS TRUE)
   available_rows
0         3611061


## 4. Data Limits

This dataset has several important limitations:

- It contains historical search performance data only and cannot explain why rankings changed.
- Some rows may contain only Google Search Console (GSC) data because Google Analytics 4 (GA4) data is unavailable.
- The analysis uses only the March 2026 snapshot and does not represent long-term trends.
- The final month (June 2026) is intentionally excluded from development because it should remain a sealed evaluation period.
- This data is useful for decision support but cannot prove cause-and-effect relationships.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=" * 60)
print("Dataset Summary")
print("=" * 60)

summary = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_content_pages,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM fact_daily;
""").df()

summary

Dataset Summary


,total_rows,total_clients,total_content_pages,gsc_available_rows,ga4_available_rows
0,9841378,55,331437,3611061.0,413966.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.